# SE-BRL Data Cleaning

## Purpose

This notebook performs reproducible data-cleaning procedures for the raw
datasets used in the SE-BRL experimental pipeline.

Cleaning decisions are derived from findings established in
`01_data_audit.ipynb`. The objective is to correct or remove documented
data-quality issues while preserving the original source information and
avoiding subjective or human-assigned annotations.

## Cleaning Principles

The following principles are applied throughout this notebook:

- Raw source files under `experiments/datasets-raw/` remain immutable.
- Every cleaning operation must be justified by an audit finding.
- No manual phishing or behavioral annotation is performed.
- Native labels are preserved until a source-supported mapping is defined.
- Spam is not automatically treated as equivalent to phishing.
- Cleaning decisions are applied programmatically and reproducibly.
- Before-and-after record counts are recorded for every operation.
- Potential grouping information required for leakage prevention is preserved.
- Model-dependent transformations are deferred to preprocessing.
- Development, calibration, and test partitions are not created or accessed
  during data cleaning.

## Inputs

Raw datasets:

`experiments/datasets-raw/`

Audit evidence:

`experiments/results/audits/`

## Outputs

Cleaned intermediate datasets will be written to:

`experiments/data/interim/`

A cleaning log will record the operations performed and their effects on
each dataset.

## Workflow

This notebook will:

1. load the audited raw datasets;
2. establish dataset-specific cleaning rules from the audit findings;
3. handle exact duplicate observations where justified;
4. address documented missing and non-finite values;
5. investigate structurally extreme email observations;
6. standardize representations required for reliable downstream processing;
7. preserve identifiers required for leakage-aware grouping;
8. validate cleaned outputs against explicit integrity checks; and
9. export cleaned intermediate datasets and cleaning documentation.

Feature scaling, model-specific encoding, train/calibration/test partitioning,
and SE-BRL behavioral feature extraction are outside the scope of this
notebook.

## 1. Environment and Paths

The cleaning environment is initialized using the same repository-relative
path structure established during the data audit. Raw datasets are loaded
from the immutable source directory, while cleaned outputs are written
separately to the intermediate-data directory.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.io import arff

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

In [4]:
# Locate repository root

cwd = Path.cwd().resolve()

REPO_ROOT = next(
    (
        p for p in [cwd, *cwd.parents]
        if (p / ".git").exists()
        and (p / "experiments").exists()
    ),
    None
)

if REPO_ROOT is None:
    raise RuntimeError(
        "Could not locate repository root."
    )

RAW_DATA_DIR = REPO_ROOT / "experiments" / "datasets-raw"
AUDIT_DIR = REPO_ROOT / "experiments" / "results" / "audits"
INTERIM_DIR = REPO_ROOT / "experiments" / "data" / "interim"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Raw data:", RAW_DATA_DIR)
print("Audit results:", AUDIT_DIR)
print("Cleaned output:", INTERIM_DIR)

Repository root: /Users/veramarie/Desktop/CS4101-Thesis2-SE-BRL
Raw data: /Users/veramarie/Desktop/CS4101-Thesis2-SE-BRL/experiments/datasets-raw
Audit results: /Users/veramarie/Desktop/CS4101-Thesis2-SE-BRL/experiments/results/audits
Cleaned output: /Users/veramarie/Desktop/CS4101-Thesis2-SE-BRL/experiments/data/interim


In [5]:
print("Raw data exists:", RAW_DATA_DIR.exists())
print("Audit results exist:", AUDIT_DIR.exists())
print("Interim output exists:", INTERIM_DIR.exists())

Raw data exists: True
Audit results exist: True
Interim output exists: True
